# Gambeli : entraîner le mot de réveil « Mésange »

**Avant de commencer** : menu *Exécution › Modifier le type d'exécution* › choisir **GPU T4**, puis *Enregistrer*.

**Ensuite, deux clics :**
1. Cliquez sur ▶ à gauche de la **première cellule** (« 1. Installation ») et attendez qu'elle se termine : la session redémarre d'elle-même, c'est normal.
2. Menu *Exécution* › **Tout exécuter**. Cela tourne seul pendant une à deux heures : gardez simplement l'onglet ouvert.

À la fin, un fichier **`mesange.zip`** se télécharge : donnez-le dans la conversation Clavarium.

*Le mot à entraîner se change dans la cellule 2 (`MOT`), par exemple pour « Gambi ».*


In [ ]:
# 1. Installation (une seule fois) -- la session redémarre ensuite d'elle-même.
import os
if os.path.exists('/content/.installed'):
    print('Déjà installé : passez à la suite.')  # évite que « Tout exécuter » redémarre la session en boucle
else:
    !pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
    !test -d microWakeWord || git clone -q https://github.com/kahrendt/microWakeWord
    !grep -q '3 \* int(1000' microWakeWord/microwakeword/data.py || sed -i 's/int(1000 \* self.step \* self.stride)/3 * int(1000 * self.step * self.stride)/' microWakeWord/microwakeword/data.py  # moins de RAM à la validation (sinon plantage)
    !pip install -q -e ./microWakeWord
    !pip install -q piper-tts==1.3.0 soundfile
    open('/content/.installed', 'w').write('ok')
    os.kill(os.getpid(), 9)   # redémarre la session : les versions installées sont alors prises en compte


In [ ]:
# 2. Le mot, et des milliers de façons de le dire, par des voix françaises de synthèse.
MOT = 'Mésange'          # le mot de réveil
NOM = 'mesange'          # le nom du fichier produit (sans accent)
DITES = [MOT, MOT + ' !', MOT + ' ?', MOT + '.', 'Mé-sange', MOT.lower()]

import os, random, wave, json, urllib.request
import numpy as np, soundfile as sf
from scipy.signal import resample_poly
from piper import PiperVoice, SynthesisConfig

assert os.path.exists('/content/.installed'), "Exécutez d'abord la cellule 1."
os.chdir('/content')
VOIX = [  # voix françaises libres de Piper (rhasspy/piper-voices) ; « mls » a 125 locuteurs
    ('fr/fr_FR/mls/medium', 'fr_FR-mls-medium'),
    ('fr/fr_FR/upmc/medium', 'fr_FR-upmc-medium'),
    ('fr/fr_FR/siwis/medium', 'fr_FR-siwis-medium'),
    ('fr/fr_FR/tom/medium', 'fr_FR-tom-medium'),
    ('fr/fr_FR/gilles/low', 'fr_FR-gilles-low'),
]
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/'
os.makedirs('voix', exist_ok=True)
for chemin, nom in VOIX:
    for ext in ('.onnx', '.onnx.json'):
        f = f'voix/{nom}{ext}'
        if not os.path.exists(f): urllib.request.urlretrieve(f'{BASE}{chemin}/{nom}{ext}', f)

os.makedirs('generated_samples', exist_ok=True)
random.seed(1)
CIBLE = 3000
n = CIBLE if len(os.listdir('generated_samples')) >= CIBLE else 0  # déjà générés : on ne refait pas
while n < CIBLE:
    for _, nom in VOIX:
        voix = PiperVoice.load(f'voix/{nom}.onnx')
        locuteurs = voix.config.num_speakers or 1
        for _ in range(CIBLE // len(VOIX) // 3 + 1):
            texte = random.choice(DITES)
            sid = random.randrange(locuteurs) if locuteurs > 1 else None
            cfg = SynthesisConfig(speaker_id=sid, length_scale=random.uniform(0.75, 1.25), noise_scale=random.uniform(0.4, 1.0), noise_w_scale=random.uniform(0.5, 1.0))
            audio = b''.join(c.audio_int16_bytes for c in voix.synthesize(texte, syn_config=cfg))
            x = np.frombuffer(audio, dtype=np.int16).astype(np.float32) / 32768
            x = resample_poly(x, 16000, voix.config.sample_rate)
            sf.write(f'generated_samples/{n}.wav', (x * 32767).astype(np.int16), 16000, subtype='PCM_16')
            n += 1
        if n >= CIBLE: break
print(n, 'exemples de', MOT)


In [ ]:
# 3. Écoutez un exemple (facultatif).
from IPython.display import Audio
Audio('generated_samples/7.wav')


In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.
# (Corrigé : décodage audio avec soundfile/librosa au lieu de torchcodec, absent de Colab.)

import io, os
import datasets
import numpy as np
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from scipy.signal import resample_poly

def to16k(x, sr):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != 16000: x = resample_poly(x, 16000, sr)
    return x

def write16k(path, x):
    sf.write(path, (np.clip(x, -1, 1) * 32767).astype(np.int16), 16000, subtype='PCM_16')

def charge(path):
    try:
        x, sr = sf.read(path, always_2d=False)
    except Exception:
        import librosa
        x, sr = librosa.load(path, sr=None, mono=True)
    return to16k(x, sr)

## Download MIT RIR data
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
if len(os.listdir(output_dir)) < 100:
    from huggingface_hub import snapshot_download
    rir_src = snapshot_download("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset")
    fichiers = [p for p in Path(rir_src).rglob("*") if p.suffix.lower() in ('.wav', '.flac', '.mp3', '.ogg')]
    print(len(fichiers), 'fichiers RIR')
    for p in tqdm(fichiers):
        try: write16k(os.path.join(output_dir, p.stem + '.wav'), charge(str(p)))
        except Exception as e: print('ignoré', p.name, e)

## Download noise and background audio
# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
os.makedirs("audioset", exist_ok=True)
output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
if len(os.listdir(output_dir)) < 100:
    # Le dépôt AudioSet est désormais en parquet : on prend une tranche (≈ 700 Mo).
    import pyarrow.parquet as pq, tempfile
    pq_path = "audioset/09.parquet"
    if not os.path.exists(pq_path):
        !wget -nv -O {pq_path} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train/09.parquet
    pf = pq.ParquetFile(pq_path)
    print(pf.schema_arrow)
    k = 0
    for batch in tqdm(pf.iter_batches(batch_size=64, columns=['audio']), total=pf.metadata.num_rows // 64 + 1):
        for a in batch.column('audio').to_pylist():
            k += 1
            try:
                data = a['bytes']; nom = os.path.basename(a.get('path') or f'clip{k}.flac')
                try:
                    x, sr = sf.read(io.BytesIO(data), always_2d=False); x = to16k(x, sr)
                except Exception:
                    with tempfile.NamedTemporaryFile(suffix=os.path.splitext(nom)[1] or '.flac') as t:
                        t.write(data); t.flush(); x = charge(t.name)
                write16k(os.path.join(output_dir, f'{k}_' + os.path.splitext(nom)[0] + '.wav'), x)
            except Exception as e:
                if k < 5: print('ignoré', k, e)

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)
output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
fname = "fma_xs.zip"
if not os.path.exists("fma/fma_small"):
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    !wget -q -O fma/{fname} {link}
    !cd fma && unzip -q -o {fname}

output_dir = "./fma_16k"
os.makedirs(output_dir, exist_ok=True)
for p in tqdm(list(Path("fma/fma_small").glob("**/*.mp3"))):
    out = os.path.join(output_dir, p.name.replace(".mp3", ".wav"))
    if os.path.exists(out): continue
    try: write16k(out, charge(str(p)))
    except Exception as e: print('ignoré', p.name, e)

print(len(os.listdir('mit_rirs')), 'RIR,', len(os.listdir('audioset_16k')), 'audioset,', len(os.listdir('fma_16k')), 'fma')


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# Fin : le modèle, prêt pour le boîtier.
import os, shutil, json
if 'NOM' not in globals(): MOT, NOM = 'Mésange', 'mesange'  # si la session a redémarré : reprendre les valeurs de la cellule 2
os.chdir('/content')
from google.colab import files
os.makedirs(NOM, exist_ok=True)
shutil.copy('trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite', f'{NOM}/{NOM}.tflite')
json.dump({'wake_word': MOT, 'micro': {'probability_cutoff': 0.9, 'sliding_window_size': 5, 'feature_step_size': 10, 'tensor_arena_size': 30000, 'minimum_esphome_version': '2024.7.0'}, 'version': 2, 'type': 'micro', 'trained_languages': ['fr']}, open(f'{NOM}/{NOM}.json', 'w'), indent=2)
!cp -r trained_models/wakeword/logs {NOM}/ 2>/dev/null || true
shutil.make_archive(NOM, 'zip', NOM)
files.download(f'{NOM}.zip')
